In [22]:
from pathlib import Path

rain_dir = Path("../data/raw/rain")

for year in range(2015, 2024):
    file_path = rain_dir / f"{year}.grd"
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"{year}: EXISTS - {size_mb:.2f} MB")
    else:
        print(f"{year}: MISSING")

2015: EXISTS - 24.25 MB
2016: EXISTS - 24.31 MB
2017: EXISTS - 24.25 MB
2018: EXISTS - 24.25 MB
2019: EXISTS - 24.25 MB
2020: EXISTS - 24.31 MB
2021: EXISTS - 24.25 MB
2022: EXISTS - 24.25 MB
2023: EXISTS - 24.25 MB


In [23]:
import imdlib as imd

data = imd.open_data('rain', 2015, 2023, 'yearwise', '../data/raw')
ds = data.get_xarray()

print("Time range:", ds.time.min().values, "to", ds.time.max().values)
print("Number of days:", ds.time.size)

Time range: 2015-01-01T00:00:00.000000 to 2023-12-31T00:00:00.000000
Number of days: 3287


In [24]:
import pandas as pd

basin = ds.sel(lat=slice(9.5, 10.5), lon=slice(76.0, 77.2))
daily_rainfall = basin["rain"].mean(dim=["lat", "lon"], skipna=True)

rain_df = daily_rainfall.to_dataframe(name="rainfall_mm").reset_index()
rain_df = rain_df.sort_values("time").reset_index(drop=True)
rain_df = rain_df[rain_df["rainfall_mm"] > -100].reset_index(drop=True)

rain_df["rainfall_3day"] = rain_df["rainfall_mm"].rolling(3).sum()
rain_df["rainfall_7day"] = rain_df["rainfall_mm"].rolling(7).sum()
rain_df["rainfall_15day"] = rain_df["rainfall_mm"].rolling(15).sum()
rain_df["rainfall_3day_max"] = rain_df["rainfall_mm"].rolling(3).max()
rain_df["rainfall_7day_max"] = rain_df["rainfall_mm"].rolling(7).max()
rain_df["rainfall_previous_day"] = rain_df["rainfall_mm"].shift(1)
rain_df["target_heavy_rain_next_day"] = (rain_df["rainfall_mm"].shift(-1) >= 64.5).astype(int)

rain_df = rain_df.dropna().reset_index(drop=True)

print(rain_df.columns.tolist())
print(rain_df.shape)
print(rain_df["target_heavy_rain_next_day"].value_counts())

out_path = Path("../data/processed/periyar_rainfall_features_full.csv")
rain_df.to_csv(out_path, index=False)
print("Saved:", out_path)

['time', 'rainfall_mm', 'rainfall_3day', 'rainfall_7day', 'rainfall_15day', 'rainfall_3day_max', 'rainfall_7day_max', 'rainfall_previous_day', 'target_heavy_rain_next_day']
(3273, 9)
target_heavy_rain_next_day
0    3237
1      36
Name: count, dtype: int64
Saved: ..\data\processed\periyar_rainfall_features_full.csv


In [25]:
check = pd.read_csv(out_path)
print("ON DISK:", check.columns.tolist())
print("ON DISK:", check.shape)

ON DISK: ['time', 'rainfall_mm', 'rainfall_3day', 'rainfall_7day', 'rainfall_15day', 'rainfall_3day_max', 'rainfall_7day_max', 'rainfall_previous_day', 'target_heavy_rain_next_day']
ON DISK: (3273, 9)
